# Egyptian TTS pipeline: test in Colab, deploy on Railway (CPU)
Runs the same service that goes to Railway: Egyptian text prep → sentence chunks → cache → **VoiceTut** → fallbacks (Edge → espeak).
Use a **CPU runtime** (Runtime → Change runtime type → CPU) to see what Railway will do. A GPU runtime also works, just faster.

**First** upload `egy-tts-pipeline.zip` in the Files pane (left sidebar), then run the cells top to bottom.

In [ ]:
import os, subprocess, textwrap
if not os.path.exists('/content/egy-tts-pipeline'):
    assert os.path.exists('/content/egy-tts-pipeline.zip'), "upload egy-tts-pipeline.zip to /content first"
    !unzip -q -o /content/egy-tts-pipeline.zip -d /content
%cd /content/egy-tts-pipeline
!ls

## 1. Install (≈3–5 min the first time)

In [ ]:
!apt-get -qq install -y espeak-ng ffmpeg libsndfile1 > /dev/null
!pip -q install git+https://github.com/k2-fsa/OmniVoice.git
!pip -q install -r requirements.txt -r requirements-model.txt

## 2. Offline tests (no model needed)
Text prep parity with your `egy.js` (needs node, preinstalled on Colab), chunking, cache, fallback and breaker logic, HTTP contract.

In [ ]:
!python -m unittest discover -s tests -t . 2>&1 | tail -5

## 3. Simulate Railway
`THREADS` = how many vCPUs your Railway service gets (Colab CPU has 2). `FORCE_CPU=True` hides the GPU even if Colab gave you one.

In [ ]:
FORCE_CPU = True
THREADS   = 2
os.environ['TORCH_THREADS'] = str(THREADS)
if FORCE_CPU: os.environ['CUDA_VISIBLE_DEVICES'] = ''
import torch; print('cuda visible to this notebook:', torch.cuda.is_available(), '| cpus:', os.cpu_count())

## 4. Benchmark the real model: pick `num_step`
RTF = seconds to generate ÷ seconds of audio. **Below 1.0 = faster than real time.** Pick the lowest step count that still sounds good in the saved samples; then set `VOICETUT_NUM_STEP` on Railway.

In [ ]:
!python tools/bench.py --steps 6 8 12 16 --save /content/bench

In [ ]:
from IPython.display import Audio, display
import glob
for f in sorted(glob.glob('/content/bench/steps*_1.wav')):
    print(f); display(Audio(f))

## 5. Start the service in the background

In [ ]:
import time, urllib.request, json
NUM_STEP = '12'          # <- from the benchmark
env = dict(os.environ, VOICETUT_NUM_STEP=NUM_STEP, PORT='8000', CACHE_DIR='/content/audio-cache', PREWARM='1', KEEPALIVE_S='240')
log = open('/content/service.log', 'w')
proc = subprocess.Popen(['python', '-m', 'uvicorn', 'egy_tts.server:app', '--port', '8000'], env=env, stdout=log, stderr=log)
for i in range(240):
    try:
        h = json.load(urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2))
        if h['ok']: break
    except Exception: pass
    time.sleep(2)
print(h); print(open('/content/service.log').read()[-1200:])

## 6. Speak

In [ ]:
import requests
from IPython.display import Audio, display
def say(text, **kw):
    t = time.time(); r = requests.post('http://127.0.0.1:8000/tts', json={'text': text, **kw}); dt = time.time() - t
    if r.status_code != 200: print(r.status_code, r.text); return
    d = int(r.headers['X-Duration-Ms']) / 1000
    print(f"{dt:.1f}s to generate {d:.1f}s of audio (RTF {dt/d:.2f}) | engine={r.headers['X-Engine']} | cache hits={r.headers['X-Cache-Hits']}")
    display(Audio(r.content))
say('أهلا بيكم في اللايف، النهارده عندنا عرض على الـ iPhone بخمسين في المية')
say('أهلا بيكم في اللايف، النهارده عندنا عرض على الـ iPhone بخمسين في المية')   # second time: straight from the cache
say('عندي meeting بكرة الساعة 3:30 والسعر 1500 جنيه', speaker='Asmaa')

## 7. Stress it like a busy chat
Eight comments at once. With `TTS_MAX_QUEUE=3` the overflow should be answered by the fallback voice (check `engine=` in the table) instead of piling up.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
def one(i):
    t = time.time(); r = requests.post('http://127.0.0.1:8000/tts', json={'text': f'تعليق رقم {i} من واحد من المشاهدين في الشات'})
    return i, r.status_code, round(time.time() - t, 1), r.headers.get('X-Engine'), r.headers.get('X-Duration-Ms')
with ThreadPoolExecutor(8) as ex:
    for row in ex.map(one, range(8)): print(row)

In [ ]:
print(json.dumps(requests.get('http://127.0.0.1:8000/metrics').json(), indent=1, ensure_ascii=False))

## 8. (Optional) point your real stream at this Colab
Opens a temporary public URL. Put it in `chat/config.js` as `VOICETUT_URL` (and `VOICETUT_TOKEN` if you set one) to test the whole avatar with this service.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
tun = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8000'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
import re
for line in tun.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m: print('VOICETUT_URL =', m.group(0)); break